# Qwen-Image-Edit-2509 Inference Benchmark

This notebook benchmarks the inference performance of the Qwen-Image-Edit-2509 model on a single Nvidia V100 GPU.

In [ ]:
from pathlib import Path
from time import perf_counter

try:
    import polars as pl
except ImportError as exc:
    raise RuntimeError("Polars is required. Run `uv sync` (or `pip install polars`) before executing this notebook.") from exc
import torch
from PIL import Image
try:
    from diffusers import QwenImageEditPlusPipeline
except ImportError as exc:
    raise RuntimeError("diffusers>=0.27.2 is required. Run `uv sync` (or `pip install diffusers>=0.27.2`).") from exc
from IPython.display import display



## Load Model and Input Images

First, we'll load the Qwen-Image-Edit-2509 pipeline and move it to the GPU. We'll also load the input images needed for the benchmark.

In [ ]:
MODEL_ID = "Qwen/Qwen-Image-Edit-2509"
pipeline = QwenImageEditPlusPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16
)
pipeline.to("cuda")
pipeline.set_progress_bar_config(disable=None)
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"Pipeline {MODEL_ID} loaded on {device_name}")

def load_image(name: str) -> Image.Image:
    search_paths = [Path("data") / name, Path(name)]
    for candidate in search_paths:
        if candidate.exists():
            return Image.open(candidate).convert("RGB")
    raise FileNotFoundError(f"Unable to find {name} in any expected location")

image1 = load_image("input1.png")
image2 = load_image("input2.png")
print("Input images loaded")


## Configure Inference Parameters

Next, we'll set up the parameters for the image editing inference.

In [ ]:
num_runs = 5
prompt = (
    "The magician bear is on the left, the alchemist bear is on the right, facing each other in the "
    "central park square."
)
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
metrics_path = results_dir / "qwen_image_edit_benchmark.parquet"
image_output_path = results_dir / "output_image_edit_plus.png"

inputs = {
    "image": [image1, image2],
    "prompt": prompt,
    "generator": torch.manual_seed(0),
    "true_cfg_scale": 4.0,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "num_images_per_prompt": 1,
}


## Run Inference Benchmark

Now we'll run the inference multiple times and measure the performance.

In [ ]:
print(f"Running inference {num_runs} times on {device_name}...")
timing_rows = []

for run_idx in range(1, num_runs + 1):
    start = perf_counter()
    with torch.inference_mode():
        output = pipeline(**inputs)
    elapsed = perf_counter() - start
    timing_rows.append({"run": run_idx, "inference_time_seconds": elapsed})
    print(f"Run {run_idx}: {elapsed:.2f} seconds")

df = pl.DataFrame(timing_rows)
display(df)

summary = df.select([
    pl.col("inference_time_seconds").mean().alias("mean_seconds"),
    pl.col("inference_time_seconds").std().alias("std_seconds"),
    pl.col("inference_time_seconds").min().alias("min_seconds"),
    pl.col("inference_time_seconds").max().alias("max_seconds"),
])
print("Benchmark summary:")
print(summary)


## Collect and Analyze Performance Metrics

Finally, we'll collect the timing results into a polars DataFrame and calculate performance statistics.

In [ ]:
output_image = output.images[0]
output_image.save(image_output_path)
print(f"Output image saved at {image_output_path.resolve()}")

df.write_parquet(metrics_path)
print(f"Timing metrics saved to {metrics_path.resolve()}")

summary_path = results_dir / "qwen_image_edit_benchmark_summary.csv"
summary.write_csv(summary_path)
print(f"Summary metrics saved to {summary_path.resolve()}")


## Save Results

We'll save both the edited image and the benchmark results to files.